## Imports

In [ ]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import seaborn

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

import PyWGCNA

## Configurações

In [ ]:
PROCESSED_DIR = Path("../../data/interim")
DESEQ_DIR = Path("../../data/interim/deseq2")
LOGCPM_PATH = PROCESSED_DIR / "microplastic_logcpm_filtered.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"
# Input: log2(normed_counts + 1) gerado pelo notebook 002
WGCNA_INPUT_PATH = PROCESSED_DIR / "microplastic_log2norm_wgcna_input.csv"

# Saídas
WGCNA_DIR = Path("../../data/interim/wgcna")
WGCNA_DIR.mkdir(parents=True, exist_ok=True)

GENE_MODULES_PATH              = WGCNA_DIR / "wgcna_gene_modules.csv"
MODULE_EIGENGENES_PATH         = WGCNA_DIR / "wgcna_module_eigengenes.csv"
MODULE_EIGENGENES_TREATED_PATH = WGCNA_DIR / "wgcna_module_eigengenes_treated.csv"
MODULE_TRAIT_CORR_PATH         = WGCNA_DIR / "wgcna_module_trait_correlations_treated.csv"
MODULE_TRAIT_PVAL_PATH         = WGCNA_DIR / "wgcna_module_trait_pvalues_treated.csv"
MODULE_TRAIT_PADJ_PATH         = WGCNA_DIR / "wgcna_module_trait_padj_treated.csv"
MODULE_SUMMARY_PATH            = WGCNA_DIR / "wgcna_module_summary.csv"
HEATMAP_PATH                   = WGCNA_DIR / "wgcna_module_trait_heatmap_treated.png"

# Cache do objeto WGCNA — apague para forçar nova execução
WGCNA_CACHE_PATH = WGCNA_DIR / "microplastic_wgcna.p"

## Carregamento dos Dados de DEG

In [ ]:
log2norm_df = pd.read_csv(WGCNA_INPUT_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("log2norm:", log2norm_df.shape)
print("metadata:", metadata_df.shape)

display(log2norm_df.head())
display(metadata_df.head())

## Construção do WGCNA

In [ ]:
# Preparação da matriz de expressão para PyWGCNA.
# A matriz em arquivo está em genes x amostras.
# Para o construtor WGCNA do PyWGCNA, deve-se usar amostras x genes.

sample_ids = metadata_df["sample_id"].tolist()
expr_sample_cols = [c for c in log2norm_df.columns if c != "gene_id"]

missing_in_expr = sorted(set(sample_ids) - set(expr_sample_cols))
missing_in_meta = sorted(set(expr_sample_cols) - set(sample_ids))

if missing_in_expr or missing_in_meta:
    raise ValueError(
        f"Inconsistência entre expressão e metadata.\n"
        f"Ausentes na expressão: {missing_in_expr}\n"
        f"Ausentes no metadata: {missing_in_meta}"
    )

# Reordena as colunas conforme a ordem do metadata
log2norm_df = log2norm_df[["gene_id"] + sample_ids]

# Formato esperado para WGCNA: amostras x genes
expr_wgcna = log2norm_df.set_index("gene_id").T
expr_wgcna.index.name = "sample_id"

# Metadata alinhado
metadata = metadata_df.set_index("sample_id").loc[expr_wgcna.index].copy()

print("Matriz para WGCNA:", expr_wgcna.shape)
print("Metadata alinhado:", metadata.shape)

display(expr_wgcna.iloc[:5, :5])
display(metadata.head())

In [ ]:
# Preparação do metadata para PyWGCNA.

# Corrige possíveis problemas de tipo ao ler CSV
if metadata["is_control"].dtype == object:
    metadata["is_control"] = metadata["is_control"].astype(str).str.lower().map({
        "true": True,
        "false": False
    })

metadata["particle_size_um"] = pd.to_numeric(metadata["particle_size_um"], errors="coerce")
metadata["particle_size_nm"] = pd.to_numeric(metadata["particle_size_nm"], errors="coerce")
metadata["concentration_gL"] = pd.to_numeric(metadata["concentration_gL"], errors="coerce")

display(metadata.dtypes)
display(metadata.head())

In [ ]:
# Construção do objeto WGCNA

pyw = PyWGCNA.WGCNA(
    name="microplastic_wgcna",          # prefixo do arquivo de cache (.p)
    species="human",                     # usado para anotação funcional (GO, KEGG)
    geneExp=expr_wgcna,                  # matriz de expressão: amostras × genes (log2norm)
    sampleInfo=metadata,                 # metadata indexado por sample_id
    outputPath=str(WGCNA_DIR) + "/",    # trailing slash obrigatório — PyWGCNA concatena strings
    save=True,                           # salva resultados intermediários automaticamente

    # Filtro de expressão mínima — já filtramos no notebook 001, então desativado aqui
    TPMcutoff=0,

    # Poderes a testar para escolha do soft-threshold β (escala 1–10 contínuo, 12–20 de 2 em 2)
    # O WGCNA seleciona o menor poder que atinge RsquaredCut na topologia livre de escala
    powers=list(range(1, 11)) + list(range(12, 22, 2)),
    RsquaredCut=0.8,                     # R² mínimo do ajuste de topologia livre de escala
    MeanCut=100,                         # conectividade média máxima (evita redes super-conectadas)

    # "signed hybrid": preserva o sinal da correlação (positivo ≠ negativo), mas trata
    # correlações negativas de forma mais suave que "signed" puro — padrão recomendado
    # para dados de RNA-seq onde co-regulação negativa é biologicamente relevante
    networkType="signed hybrid",
    TOMType="signed",                    # TOM com sinal: considera direção da correlação no overlap

    # Módulo com menos de 50 genes é dissolvido e seus genes redistribuídos
    # Valor conservador — reduz ruído, mas pode aumentar o dimgrey (genes não atribuídos)
    minModuleSize=50,

    # Módulos com correlação entre eigengenes > 1 - 0.2 = 0.8 são fundidos
    # Valor baixo (0.2) = fusão mais agressiva → menos módulos, mais coesos
    MEDissThres=0.2,
)

In [ ]:
if WGCNA_CACHE_PATH.exists():
    print(f"Carregando WGCNA do cache: {WGCNA_CACHE_PATH}")
    # pyw = PyWGCNA.readWGCNA("microplastic_wgcna", outputPath=str(WGCNA_DIR))
    with open(WGCNA_CACHE_PATH, "rb") as f:
        pyw = pickle.load(f)
    print("WGCNA carregado com sucesso.")
else:
    print("Cache não encontrado — executando WGCNA (pode levar vários minutos)...")
    pyw.runWGCNA()
    if hasattr(pyw, "saveWGCNA"):
        pyw.saveWGCNA()
    print(f"WGCNA executada e cache salvo em: {WGCNA_CACHE_PATH}")

## Inspeção dos Resultados

In [ ]:
# Verificar os principais atributos do objeto PyWGCNA após a execução

print("type(pyw.datExpr):", type(getattr(pyw, "datExpr", None)))
print("shape datExpr:", getattr(getattr(pyw, "datExpr", None), "shape", None))

if hasattr(pyw, "datExpr") and pyw.datExpr is not None:
    print("var columns:", pyw.datExpr.var.columns)
    display(pyw.datExpr.var.head())

In [ ]:
# Extração de genes e módulos finais obtidos com WGCNA

if not hasattr(pyw, "datExpr") or pyw.datExpr is None:
    raise ValueError("pyw.datExpr não está disponível.")

gene_var = pyw.datExpr.var.copy()

expected_cols = {"dynamicColors", "moduleColors", "moduleLabels"}
missing_cols = expected_cols - set(gene_var.columns)

if missing_cols:
    raise ValueError(f"Colunas esperadas ausentes em pyw.datExpr.var: {missing_cols}")

gene_modules_df = (
    gene_var
    .reset_index()
    .rename(columns={"index": "gene_id"})
    [["gene_id", "dynamicColors", "moduleColors", "moduleLabels"]]
    .copy()
)

gene_modules_df = gene_modules_df.rename(columns={
    "dynamicColors": "dynamic_module",
    "moduleColors": "module",
    "moduleLabels": "module_label"
})

gene_modules_df.to_csv(GENE_MODULES_PATH, index=False)

print("Tabela gene -> módulo salva em:", GENE_MODULES_PATH)
print("Dimensões:", gene_modules_df.shape)

display(gene_modules_df.head())
display(
    gene_modules_df["module"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "module", "module": "n_genes"})
)

# NOTA: O módulo 'dimgrey' representa genes não atribuídos a nenhum módulo de
# coexpressão (equivalente ao módulo "grey" do WGCNA em R). Um valor acima de
# 20-25% do total sugere que os parâmetros podem ser ajustados para capturar mais
# estrutura (ex: menor minModuleSize, menor MEDissThres, ou outro soft-threshold power).
n_dimgrey = (gene_modules_df["module"] == "dimgrey").sum()
n_total = len(gene_modules_df)
pct_unassigned = 100 * n_dimgrey / n_total
print(f"\n⚠️ Módulo 'dimgrey' (não atribuídos): {n_dimgrey} genes ({pct_unassigned:.1f}% do total)")
print("Referência: < 20-25% é considerado aceitável. Considere ajustar minModuleSize, MEDissThres ou power.")

In [ ]:
# Cálculo dos module eigengenes a partir dos módulos finais.
#
# NOTA: Calculamos eigengenes manualmente via PCA ao invés de usar pyw.getEigengenes()
# porque a API do PyWGCNA retorna eigengenes na orientação interna do objeto AnnData,
# dificultando o alinhamento com nosso metadata indexado por sample_id.
# O resultado é matematicamente equivalente: o 1º componente principal do bloco de
# expressão de cada módulo, com sinal corrigido para correlacionar positivamente com
# o perfil médio do módulo.

module_eigengenes = {}

for module_name, module_df in gene_modules_df.groupby("module"):
    genes = [g for g in module_df["gene_id"].tolist() if g in expr_wgcna.columns]

    if len(genes) == 0:
        continue

    X = expr_wgcna[genes].copy()

    # Se o módulo tiver apenas 1 gene, usa o próprio perfil
    if X.shape[1] == 1:
        eigengene = X.iloc[:, 0].values.astype(float)
    else:
        pca = PCA(n_components=1, random_state=42)
        eigengene = pca.fit_transform(X.values).ravel()

        # Ajusta o sinal para ficar coerente com o perfil médio do módulo
        mean_profile = X.mean(axis=1).values
        corr_sign = np.corrcoef(eigengene, mean_profile)[0, 1]
        if pd.notna(corr_sign) and corr_sign < 0:
            eigengene = -eigengene

    module_eigengenes[f"ME_{module_name}"] = eigengene

module_eigengenes_df = pd.DataFrame(
    module_eigengenes,
    index=expr_wgcna.index
)

module_eigengenes_df.index.name = "sample_id"
module_eigengenes_df.to_csv(MODULE_EIGENGENES_PATH)

print("Module eigengenes salvos em:", MODULE_EIGENGENES_PATH)
print("Dimensões:", module_eigengenes_df.shape)

display(module_eigengenes_df.head())

In [ ]:
# Filtrar apenas as amostras tratadas para correlação módulo-traço

metadata_treated = metadata[metadata["is_control"] == False].copy()
module_eigengenes_treated = module_eigengenes_df.loc[metadata_treated.index].copy()

print("Amostras totais:", metadata.shape[0])
print("Amostras tratadas:", metadata_treated.shape[0])
print("Module eigengenes (tratadas):", module_eigengenes_treated.shape)

display(metadata_treated.head())
display(module_eigengenes_treated.head())

In [ ]:
# Montar traços de interesse para correlação

metadata_treated = metadata_treated.copy()

# Variável binária principal para distinguir 100 nm de 1 µm
metadata_treated["is_100nm"] = (metadata_treated["particle_size_nm"] == 100).astype(int)

# Dummies por grupo para leitura mais fina no heatmap
group_dummies_treated = pd.get_dummies(metadata_treated["group"], prefix="group")

# Traits não redundantes
traits_numeric_treated = pd.concat([
    metadata_treated[[
        "particle_size_um",
        "concentration_gL",
        "is_100nm",
    ]],
    group_dummies_treated
], axis=1)

print("Traits numéricos (tratadas):", traits_numeric_treated.shape)
display(traits_numeric_treated.head())

In [ ]:
# Correlação módulo-traço

def safe_pearsonr(x, y):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)

    mask = x.notna() & y.notna()

    if mask.sum() < 3:
        return np.nan, np.nan

    if x[mask].nunique() < 2 or y[mask].nunique() < 2:
        return np.nan, np.nan

    r, p = pearsonr(x[mask], y[mask])
    return r, p

corr_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

pval_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

for me_col in module_eigengenes_treated.columns:
    for trait_col in traits_numeric_treated.columns:
        r, p = safe_pearsonr(
            module_eigengenes_treated[me_col],
            traits_numeric_treated[trait_col]
        )
        corr_matrix_treated.loc[me_col, trait_col] = r
        pval_matrix_treated.loc[me_col, trait_col] = p

print("Matriz de correlação:", corr_matrix_treated.shape)
print("Matriz de p-values:", pval_matrix_treated.shape)

display(corr_matrix_treated)
display(pval_matrix_treated)

In [ ]:
# Correção para múltiplos testes: Benjamini-Hochberg (FDR) sobre os 140 testes
# simultâneos (14 módulos × 10 traits). Sem correção, ~7 associações falsas seriam
# esperadas ao nível α=0.05 apenas por acaso.

padj_matrix_treated = pval_matrix_treated.copy().astype(float)

for trait_col in pval_matrix_treated.columns:
    pvals = pval_matrix_treated[trait_col].values.astype(float)
    valid_mask = ~np.isnan(pvals)
    if valid_mask.sum() > 0:
        _, padj, _, _ = multipletests(pvals[valid_mask], method="fdr_bh")
        padj_col = np.full(len(pvals), np.nan)
        padj_col[valid_mask] = padj
        padj_matrix_treated[trait_col] = padj_col

print("Matriz de p-valores ajustados (BH FDR):", padj_matrix_treated.shape)
display(padj_matrix_treated)

In [ ]:
fig = px.imshow(
    corr_matrix_treated.astype(float),
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect="auto",
    labels=dict(x="Traits", y="Módulos", color="Correlação"),
    x=corr_matrix_treated.columns,
    y=corr_matrix_treated.index,
    title="Correlação entre module eigengenes e traits (somente tratadas)"
)

fig.update_layout(
    width=900, height=max(400, 35*len(corr_matrix_treated)),
    margin=dict(l=100, r=20, t=60, b=100),
    xaxis_title="Traits",
    yaxis_title="Módulos"
)

fig.show()

# Salva o heatmap como PNG (requer kaleido: pip install kaleido)
fig.write_image(str(HEATMAP_PATH))
print("Heatmap salvo em:", HEATMAP_PATH)

## Consolidação dos Resultados do WGCNA

In [ ]:
# Salvamento dos resultados sem controles

module_eigengenes_treated.to_csv(MODULE_EIGENGENES_TREATED_PATH)
corr_matrix_treated.to_csv(MODULE_TRAIT_CORR_PATH)
pval_matrix_treated.to_csv(MODULE_TRAIT_PVAL_PATH)
padj_matrix_treated.to_csv(MODULE_TRAIT_PADJ_PATH)

print("Arquivos salvos:")
print("-", MODULE_EIGENGENES_TREATED_PATH)
print("-", MODULE_TRAIT_CORR_PATH)
print("-", MODULE_TRAIT_PVAL_PATH)
print("-", MODULE_TRAIT_PADJ_PATH)

In [ ]:
# Resumo dos módulos mais associados aos traços de interesse.
# Inclui correlação, p-valor bruto e p-valor ajustado por BH (FDR).
# O flag `significant_*` usa padj < 0.05.

summary_rows = []

for module_name in corr_matrix_treated.index:
    row = {"module": module_name}

    for trait in ["particle_size_um", "concentration_gL", "is_100nm"]:
        if trait in corr_matrix_treated.columns:
            row[f"corr_{trait}"]        = corr_matrix_treated.loc[module_name, trait]
            row[f"pval_{trait}"]        = pval_matrix_treated.loc[module_name, trait]
            row[f"padj_{trait}"]        = padj_matrix_treated.loc[module_name, trait]
            row[f"significant_{trait}"] = padj_matrix_treated.loc[module_name, trait] < 0.05

    summary_rows.append(row)

module_summary_df = pd.DataFrame(summary_rows)
module_summary_df.to_csv(MODULE_SUMMARY_PATH, index=False)

print("Resumo dos módulos salvo em:", MODULE_SUMMARY_PATH)
display(module_summary_df.sort_values("pval_particle_size_um", na_position="last").head(15))